In [15]:

from __future__ import annotations
from typing import Annotated, TypedDict, List
from datetime import datetime, timezone, timedelta
from dotenv import load_dotenv
import os
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage, BaseMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

load_dotenv()

True

## 定义知识库

In [2]:
# # ============== 2.1 迷你知识库（演示检索用，无需联网） ==============
DOCS = [
    {
        "title": "Python",
        "content": (
            "Python 是一种高级编程语言，由 Guido van Rossum 创建，"
            "以简洁与可读性著称，拥有庞大的生态和社区。"
        ),
        "keywords": ["python", "guido", "rossum", "编程语言"],
    },
    {
        "title": "LangGraph",
        "content": (
            "LangGraph 是 LangChain 团队开发的框架，用图（状态机）来编排多步骤智能体，"
            "支持节点、边、分支、循环、流式、人工介入等特性。"
        ),
        "keywords": ["langgraph", "状态机", "agent", "图"],
    },
    {
        "title": "ReAct 模式",
        "content": (
            "ReAct 是一种让模型在推理时交替进行 '思考' 与 '行动' 的范式："
            "模型先决定是否调用工具获取信息（行动），再根据观察结果继续推理，"
            "直至得到最终答案。"
        ),
        "keywords": ["react", "思考", "行动", "工具调用"],
    },
]


## 工具实现

In [29]:
def _safe_calculate(expr: str):
    """
    安全计算表达式，仅允许使用 sqrt, log, log10, sin, cos, tan 等函数
    """

    # 1. 白名单检查：只能出现数字、运算符、小数点、括号和允许的函数名
    allowed_chars = "0123456789+-*/()., %"
    allowed_funcs = {
        "sqrt": math.sqrt,
        "log": math.log,
        "log10": math.log10,
        "sin": math.sin,
        "cos": math.cos,
        "tan": math.tan,
    }
    
    # 检查是否包含非法字符
    for ch in expr:
        if not (ch in allowed_chars or ch.isalpha()):
            raise ValueError("表达式包含非法字符")
    
    # 2. 在受限环境下执行 eval
    try:
        result = eval(expr, {"__builtins__": None}, allowed_funcs)
    except Exception as e:
        raise ValueError(f"计算出错: {e}")
    
    return result


def _score(q: str, doc: dict) -> int:
    """极简打分：按关键字词频/包含数计分"""
    qlow = q.lower()
    score = 0
    for kw in doc["keywords"]:
        if kw.lower() in qlow:
            score += 2
    for w in qlow.split():
        if w in doc["content"].lower():
            score += 1
    return score

_safe_calculate("sqrt(9)"), _score("sqrt(9)", {"keywords": ["sqrt"], "content": "这是一个关于平方根的文档"})

(3.0, 2)

In [30]:
@tool
def search_docs(query: str, top_k: int = 3) -> str:
    """迷你文档检索：在内置的小型知识库中按关键词匹配并返回前 top_k 条摘要。参数：query, top_k=3。"""
    try:
        print("**正在调用文档检索工具**")
        scored = sorted(DOCS, key=lambda d: _score(query, d), reverse=True)[: int(top_k)]
        if not scored:
            return "未找到相关文档"
        lines = []
        for i, d in enumerate(scored, 1):
            snippet = d["content"][:160]
            lines.append(f"[{i}] {d['title']}: {snippet}")
        return "\n".join(lines)
    except Exception as e:
        return f"搜索失败: {e}"
    
@tool
def calculate(expr: str) -> str:
    """安全计算表达式，仅允许使用 sqrt, log, log10, sin, cos, tan 等函数。参数：expr。"""
    try:
        print("**正在调用计算工具**")
        result = _safe_calculate(expr)
        return f"计算结果: {result}"
    except ValueError as e:
        return f"计算失败: {e}"
    
@tool
def get_current_time(tz: str = "UTC") -> str:
    """获取指定时区的当前时间。参数：tz="UTC"。"""
    try:
        print("**正在调用时间工具**")
        return f"当前时间: {datetime.now(timezone(tz)).strftime('%Y-%m-%d %H:%M:%S %Z')}"
    except ValueError as e:
        return f"获取时间失败: {e}"


In [17]:
# 工具列表
TOOLS = [calculate, search_docs, get_current_time]


## 定义state

In [18]:
class AgentState(TypedDict):
    # 使用 add_messages 归约器：节点返回的消息会“追加”到历史里，而不是覆盖
    messages: Annotated[List[BaseMessage], add_messages]
    # 也可以扩展其他字段，比如计数器、临时变量等
    # loops: int

## 构建LLM

In [19]:
llm = ChatOpenAI(
    model=os.getenv("MODEL_NAME"),
    base_url=os.getenv("BASE_URL"),
    api_key=os.getenv("OPENAI_API_KEY"),
    temperature=0,
).bind_tools(TOOLS)

## prompt

In [20]:
SYSTEM_PROMPT = (
    "你是一个基于 ReAct 模式的助理。"
    "当问题需要计算或事实检索、当前时间时，合理调用工具；"
    "得到工具结果后，整合并给出结论。"
    "回答要准确、简洁、可复核；不要编造工具不存在的能力。"
)

In [24]:
# agent 节点：让模型决定是否调用工具（函数调用）
def agent(state: AgentState) -> AgentState:
    msgs = state["messages"]
    # 注入 system 提示
    inputs = [{"role": "system", "content": SYSTEM_PROMPT}] + [m for m in msgs]
    ai_res = llm.invoke(inputs)
    return {"messages": [ai_res]}


In [21]:
tool_node = ToolNode(TOOLS)

In [22]:
# ============== 2.5 条件路由：有 tool_calls → 去 tools，否则结束 ==============
def should_continue(state: AgentState) -> str:
    last = state["messages"][-1]
    if isinstance(last, AIMessage) and getattr(last, "tool_calls", None):
        return "tools"
    return "end"

## 组合图

In [25]:
builder = StateGraph(AgentState)
builder.add_node("agent", agent)
builder.add_node("tools", tool_node)

# 添加边 start -> agent
builder.set_entry_point("agent")    

builder.add_conditional_edges(
    "agent",
    should_continue,
    {
        "tools": "tools",
        "end": END,
    },
)
builder.add_edge("tools", "agent")

In [26]:
graph = builder.compile()

In [31]:
questions = [
    # 需要检索的问题
    "Python 的创始人是谁？给一句简要说明。",
    # 需要计算的问题
    "如果太阳到地球平均距离约 1.496e8 公里，光速约 3e5 公里/秒，光需要多少分钟到达地球？只给结果到 2 位小数。",
    # 需要当前时间
    "现在的 UTC+8 时间是多少？",
    # 混合问题（检索+解释）
    "LangGraph 是什么？一句话讲清，再给一个使用场景。",
]

for q in questions:
    print("\n=== 用户:", q)
    state = {
        "messages": [HumanMessage(content=q)]
    }
    # 也可以用 stream 查看过程事件（节点执行顺序）
    # for event in graph.stream(state, stream_mode="values"):
    #     print("  [事件]", type(event["messages"][-1]).__name__, "->", event["messages"][-1].content)

    out = graph.invoke(state)
    final_msg = out["messages"][-1]
    print(">>> 答案:", final_msg.content)
    print("\n============================\n")



=== 用户: Python 的创始人是谁？给一句简要说明。
>>> 答案: Python 由 Guido van Rossum 创建，是一种以简洁与可读性著称的高级编程语言。



=== 用户: 如果太阳到地球平均距离约 1.496e8 公里，光速约 3e5 公里/秒，光需要多少分钟到达地球？只给结果到 2 位小数。
>>> 答案: 光从太阳到地球需要约 8.31 分钟到达地球。



=== 用户: 现在的 UTC+8 时间是多少？
>>> 答案: 看起来我在获取当前时间时遇到了时区参数的问题。我将直接提供当前时间的标准 UTC 格式，并说明时区转换需要额外处理。

当前 UTC 时间为：2023-10-04 12:00:00（示例时间，具体以实际时间为准）。  
如需 UTC+8 时间，请将上述时间加 8 小时，即为：2023-10-04 20:00:00（示例时间）。



=== 用户: LangGraph 是什么？一句话讲清，再给一个使用场景。
>>> 答案: LangGraph 是 LangChain 团队开发的框架，用图（状态机）来编排多步骤智能体，支持节点、边、分支、循环、流式、人工介入等特性。

使用场景：可以用于构建复杂的 AI 应用流程，例如自动化客服系统，在其中定义多个智能体节点来处理用户问题、转接人工服务或触发外部 API 查询。


